# Application Tunnel + Rovo MCP Integration Test

This notebook tests the integration of on-premise Confluence (Data Center) with Atlassian's MCP server via Application Tunnels and Rovo.

## Architecture

```
┌─────────────────┐
│   MCP Client    │
│   (Claude)      │
└────────┬────────┘
         │ MCP Protocol (JSON-RPC over HTTPS)
         ▼
┌─────────────────┐
│ Atlassian MCP   │  mcp.atlassian.com
│ Server (Cloud)  │
└────────┬────────┘
         │ Internal Cloud APIs
         ▼
┌─────────────────┐
│ Rovo AI Engine  │  Indexes content from connected sources
│ (Cloud)         │  Respects Data Center permissions
└────────┬────────┘
         │ Application Tunnel (Encrypted)
         ▼
┌─────────────────┐     ┌─────────────────┐
│ Tunnel Client   │ ◄─► │ Confluence DC   │
│ (Marketplace)   │     │ (On-premise)    │
└─────────────────┘     └─────────────────┘
```

## Prerequisites

This integration requires:
1. **Atlassian Cloud subscription** (Premium or Enterprise tier)
2. **Application Tunnel** configured between Cloud and Data Center
3. **Rovo** enabled and configured to index Data Center content
4. **MCP Server** access enabled in Rovo settings

## References
- [Application Tunnels Documentation](https://support.atlassian.com/organization-administration/docs/connect-to-self-managed-products-with-application-tunnels/)
- [Atlassian MCP Server](https://www.atlassian.com/platform/remote-mcp-server)
- [Rovo Documentation](https://www.atlassian.com/software/rovo)

## 1. Prerequisites Checklist

Complete this checklist before proceeding. Items marked ❌ will block testing.

In [ ]:
# Prerequisites Checklist
# Update these values based on your organization's setup

prerequisites = {
    "cloud_subscription": {
        "name": "Atlassian Cloud Subscription",
        "status": False,  # Set to True when acquired
        "details": "Premium or Enterprise tier required for Rovo",
        "action": "Contact IT/Procurement to acquire Cloud subscription",
        "owner": "IT/Procurement",
    },
    "cloud_site_url": {
        "name": "Cloud Site URL Known",
        "status": False,  # Set to True when URL is known
        "details": "e.g., https://abbvie.atlassian.net",
        "action": "Get Cloud site URL from Cloud Admin",
        "owner": "Cloud Admin",
    },
    "tunnel_marketplace_app": {
        "name": "Tunnel Marketplace App Installed on DC",
        "status": False,  # Set to True when installed
        "details": "'Application Tunnels' app from Atlassian Marketplace",
        "action": "Install app on Confluence Data Center",
        "owner": "DC Admin",
    },
    "tunnel_created": {
        "name": "Application Tunnel Created",
        "status": False,  # Set to True when tunnel is created
        "details": "Tunnel created in admin.atlassian.com",
        "action": "Create tunnel and generate security key",
        "owner": "Cloud Admin",
    },
    "tunnel_connected": {
        "name": "Tunnel Connected and Healthy",
        "status": False,  # Set to True when tunnel is connected
        "details": "Security key added to DC, tunnel shows 'Connected'",
        "action": "Add security key to DC tunnel client",
        "owner": "DC Admin",
    },
    "rovo_enabled": {
        "name": "Rovo Enabled on Cloud",
        "status": False,  # Set to True when Rovo is enabled
        "details": "Rovo AI features activated in Cloud settings",
        "action": "Enable Rovo in Atlassian Cloud admin",
        "owner": "Cloud Admin",
    },
    "rovo_dc_connector": {
        "name": "Rovo Connected to Data Center",
        "status": False,  # Set to True when DC is a Rovo source
        "details": "Data Center added as a content source in Rovo",
        "action": "Configure Rovo to index DC content via tunnel",
        "owner": "Cloud Admin",
    },
    "rovo_indexing_complete": {
        "name": "Rovo Indexing Complete",
        "status": False,  # Set to True when indexing is done
        "details": "DC content appears in Rovo search results",
        "action": "Wait for initial indexing to complete",
        "owner": "Automatic",
    },
    "mcp_enabled": {
        "name": "MCP Server Enabled",
        "status": False,  # Set to True when MCP is enabled
        "details": "MCP server access enabled in Rovo settings",
        "action": "Enable MCP server in admin settings",
        "owner": "Cloud Admin",
    },
    "api_token_generated": {
        "name": "Cloud API Token Generated",
        "status": False,  # Set to True when token is ready
        "details": "API token for Cloud (different from DC token)",
        "action": "Generate API token in Atlassian account settings",
        "owner": "User",
    },
}

# Display checklist
print("=" * 70)
print("PREREQUISITES CHECKLIST")
print("=" * 70)
print()

all_ready = True
blocking_items = []

for key, item in prerequisites.items():
    status_icon = "✅" if item["status"] else "❌"
    print(f"{status_icon} {item['name']}")
    print(f"   Details: {item['details']}")
    if not item["status"]:
        print(f"   Action: {item['action']}")
        print(f"   Owner: {item['owner']}")
        all_ready = False
        blocking_items.append(item)
    print()

print("=" * 70)
if all_ready:
    print("✅ ALL PREREQUISITES MET - Ready to test MCP integration")
else:
    print(f"❌ {len(blocking_items)} PREREQUISITES NOT MET")
    print()
    print("Action Items for IT/Admin:")
    for i, item in enumerate(blocking_items, 1):
        print(f"  {i}. [{item['owner']}] {item['action']}")
print("=" * 70)

## 2. Configuration

Configure credentials for both Cloud and Data Center access.

In [ ]:
import sys
sys.path.append('..')

import os
import json
import base64
import asyncio
import requests
import warnings
from datetime import datetime
from typing import Optional, Dict, Any, List
from dataclasses import dataclass
from urllib3.exceptions import InsecureRequestWarning

# Suppress SSL warnings for corporate proxy
warnings.filterwarnings('ignore', category=InsecureRequestWarning)

from dotenv import load_dotenv
load_dotenv()

# Check for MCP SDK
try:
    import httpx
    from mcp import ClientSession
    from mcp.client.streamable_http import streamable_http_client
    MCP_AVAILABLE = True
    print("✅ MCP SDK available")
except ImportError as e:
    MCP_AVAILABLE = False
    print(f"❌ MCP SDK not available: {e}")

print()

In [ ]:
@dataclass
class TunnelMCPConfig:
    """Configuration for Application Tunnel + MCP integration."""
    
    # Atlassian Cloud (required for MCP)
    cloud_url: Optional[str] = None  # e.g., https://abbvie.atlassian.net
    cloud_email: Optional[str] = None
    cloud_api_token: Optional[str] = None
    
    # MCP Server
    mcp_endpoint: str = "https://mcp.atlassian.com/v1/mcp"
    protocol_version: str = "2025-06-18"
    
    # Data Center (for comparison testing)
    dc_url: Optional[str] = None  # e.g., https://confluence.abbvienet.com
    dc_username: Optional[str] = None
    dc_api_token: Optional[str] = None
    
    @classmethod
    def from_env(cls) -> "TunnelMCPConfig":
        """Load configuration from environment variables."""
        return cls(
            # Cloud credentials
            cloud_url=os.getenv("ATLASSIAN_CLOUD_URL"),
            cloud_email=os.getenv("ATLASSIAN_CLOUD_EMAIL") or os.getenv("ATLASSIAN_USER_EMAIL"),
            cloud_api_token=os.getenv("ATLASSIAN_CLOUD_API_TOKEN"),
            # Data Center credentials (existing)
            dc_url=os.getenv("CONFLUENCE_URL") or os.getenv("ATLASSIAN_SITE_URL"),
            dc_username=os.getenv("CONFLUENCE_USERNAME"),
            dc_api_token=os.getenv("CONFLUENCE_API_TOKEN"),
        )
    
    def has_cloud_config(self) -> bool:
        """Check if Cloud credentials are configured."""
        return bool(self.cloud_url and self.cloud_email and self.cloud_api_token)
    
    def has_dc_config(self) -> bool:
        """Check if Data Center credentials are configured."""
        return bool(self.dc_url and self.dc_username and self.dc_api_token)
    
    def get_cloud_auth_header(self) -> Optional[str]:
        """Get Authorization header for Cloud API."""
        if not self.has_cloud_config():
            return None
        credentials = f"{self.cloud_email}:{self.cloud_api_token}"
        encoded = base64.b64encode(credentials.encode()).decode()
        return f"Basic {encoded}"

# Load configuration
config = TunnelMCPConfig.from_env()

print("=" * 70)
print("CONFIGURATION STATUS")
print("=" * 70)
print()
print("Atlassian Cloud:")
print(f"  URL: {config.cloud_url or 'NOT SET'}")
print(f"  Email: {config.cloud_email or 'NOT SET'}")
print(f"  API Token: {'*' * 8 + config.cloud_api_token[-4:] if config.cloud_api_token else 'NOT SET'}")
print(f"  Status: {'✅ Configured' if config.has_cloud_config() else '❌ Not configured'}")
print()
print("Data Center (for comparison):")
print(f"  URL: {config.dc_url or 'NOT SET'}")
print(f"  Username: {config.dc_username or 'NOT SET'}")
print(f"  API Token: {'*' * 8 + config.dc_api_token[-4:] if config.dc_api_token else 'NOT SET'}")
print(f"  Status: {'✅ Configured' if config.has_dc_config() else '❌ Not configured'}")
print()
print("MCP Server:")
print(f"  Endpoint: {config.mcp_endpoint}")
print("=" * 70)

if not config.has_cloud_config():
    print()
    print("⚠️  Cloud credentials not configured!")
    print("   Add these environment variables to .env:")
    print("   ATLASSIAN_CLOUD_URL=https://yourcompany.atlassian.net")
    print("   ATLASSIAN_CLOUD_EMAIL=your.email@company.com")
    print("   ATLASSIAN_CLOUD_API_TOKEN=your_cloud_token")

## 3. Cloud MCP Connection Test

Test connection to the Atlassian MCP server using Cloud credentials.

**Expected behavior with proper Cloud + Tunnel + Rovo setup:**
- `initialize` should succeed
- `tools/list` should return available tools
- Tools should include Confluence search capabilities

In [ ]:
def build_mcp_headers(config: TunnelMCPConfig) -> Dict[str, str]:
    """Build headers for MCP requests using Cloud credentials."""
    headers = {
        "Content-Type": "application/json",
        "Accept": "application/json, text/event-stream",
        "MCP-Protocol-Version": config.protocol_version,
    }
    
    auth = config.get_cloud_auth_header()
    if auth:
        headers["Authorization"] = auth
    
    return headers

def send_mcp_request(
    endpoint: str,
    method: str,
    params: Dict[str, Any],
    headers: Dict[str, str],
    session_id: Optional[str] = None,
    request_id: int = 1,
) -> Dict[str, Any]:
    """Send an MCP request and parse the response."""
    
    if session_id:
        headers = {**headers, "Mcp-Session-Id": session_id}
    
    request = {
        "jsonrpc": "2.0",
        "id": request_id,
        "method": method,
        "params": params,
    }
    
    try:
        response = requests.post(
            endpoint,
            json=request,
            headers=headers,
            timeout=30,
            stream=True,
            verify=False,
        )
        
        # Extract session ID from response headers
        new_session_id = response.headers.get("Mcp-Session-Id")
        
        content_type = response.headers.get("Content-Type", "")
        
        if "text/event-stream" in content_type:
            # Parse SSE response
            for line in response.iter_lines(decode_unicode=True):
                if line and line.startswith("data:"):
                    data = line[5:].strip()
                    try:
                        parsed = json.loads(data)
                        return {
                            "success": True,
                            "response": parsed,
                            "session_id": new_session_id,
                        }
                    except json.JSONDecodeError:
                        return {"success": False, "error": f"Invalid JSON: {data}"}
            return {"success": False, "error": "No data in SSE response"}
        
        elif "application/json" in content_type:
            return {
                "success": True,
                "response": response.json(),
                "session_id": new_session_id,
            }
        
        else:
            return {
                "success": False,
                "error": f"Unexpected content type: {content_type}",
                "body": response.text[:500],
            }
            
    except Exception as e:
        return {"success": False, "error": str(e)}

print("MCP request functions defined.")

In [ ]:
# Test MCP connection with Cloud credentials
print("=" * 70)
print("MCP CONNECTION TEST (Cloud Credentials)")
print("=" * 70)
print()

if not config.has_cloud_config():
    print("❌ Cannot test - Cloud credentials not configured")
    print("   Configure ATLASSIAN_CLOUD_* environment variables first")
else:
    headers = build_mcp_headers(config)
    
    # Step 1: Initialize session
    print("Step 1: Initialize MCP session...")
    init_result = send_mcp_request(
        config.mcp_endpoint,
        "initialize",
        {
            "protocolVersion": config.protocol_version,
            "capabilities": {"tools": {}, "resources": {}, "prompts": {}},
            "clientInfo": {"name": "tunnel-mcp-test", "version": "1.0.0"}
        },
        headers
    )
    
    if init_result["success"]:
        session_id = init_result.get("session_id")
        server_info = init_result["response"].get("result", {}).get("serverInfo", {})
        capabilities = init_result["response"].get("result", {}).get("capabilities", {})
        
        print(f"  ✅ Session initialized")
        print(f"     Session ID: {session_id}")
        print(f"     Server: {server_info}")
        print(f"     Capabilities: {list(capabilities.keys())}")
        print()
        
        # Step 2: List tools
        print("Step 2: List available tools...")
        tools_result = send_mcp_request(
            config.mcp_endpoint,
            "tools/list",
            {},
            headers,
            session_id=session_id,
            request_id=2
        )
        
        if tools_result["success"]:
            response = tools_result["response"]
            if "error" in response:
                print(f"  ❌ Error: {response['error'].get('message', 'Unknown')}")
                print()
                print("  This likely means:")
                print("  - Rovo is not enabled, OR")
                print("  - MCP server access is not enabled, OR")
                print("  - Application Tunnel is not configured")
            else:
                tools = response.get("result", {}).get("tools", [])
                print(f"  ✅ Found {len(tools)} tools")
                for tool in tools:
                    print(f"     - {tool.get('name')}: {tool.get('description', 'No description')[:60]}")
        else:
            print(f"  ❌ Failed: {tools_result.get('error')}")
    else:
        print(f"  ❌ Initialization failed: {init_result.get('error')}")

print()
print("=" * 70)

## 4. Data Center Content Discovery

If the tunnel is properly configured, search for content that exists in Data Center.

**Test Strategy:**
1. Choose a page that only exists in Data Center
2. Search for it via MCP (which queries Rovo's index)
3. If found, the tunnel is working

In [ ]:
# Configure test queries
# Update these with actual page titles/content from your Data Center

DC_TEST_QUERIES = [
    "DSA team documentation",  # Replace with actual DC content
    "internal project wiki",
    "data science architecture",
]

print("Test queries configured:")
for i, q in enumerate(DC_TEST_QUERIES, 1):
    print(f"  {i}. {q}")
print()
print("Update DC_TEST_QUERIES with actual page titles from your Data Center")
print("to verify tunnel connectivity.")

In [ ]:
# Test searching for DC content via MCP
print("=" * 70)
print("DATA CENTER CONTENT DISCOVERY VIA MCP")
print("=" * 70)
print()

if not config.has_cloud_config():
    print("❌ Cannot test - Cloud credentials not configured")
else:
    headers = build_mcp_headers(config)
    
    # Initialize session
    init_result = send_mcp_request(
        config.mcp_endpoint,
        "initialize",
        {
            "protocolVersion": config.protocol_version,
            "capabilities": {"tools": {}},
            "clientInfo": {"name": "dc-discovery-test", "version": "1.0.0"}
        },
        headers
    )
    
    if not init_result["success"]:
        print(f"❌ Failed to initialize: {init_result.get('error')}")
    else:
        session_id = init_result.get("session_id")
        print(f"Session established: {session_id[:20]}...")
        print()
        
        # Try different tool names for searching
        search_tool_names = [
            "confluence_search",
            "search",
            "confluence-search",
            "searchConfluence",
            "rovo_search",
        ]
        
        for query in DC_TEST_QUERIES:
            print(f"Searching for: '{query}'")
            
            found_tool = False
            for tool_name in search_tool_names:
                result = send_mcp_request(
                    config.mcp_endpoint,
                    "tools/call",
                    {"name": tool_name, "arguments": {"query": query}},
                    headers,
                    session_id=session_id,
                )
                
                if result["success"]:
                    response = result["response"]
                    if "error" not in response:
                        found_tool = True
                        print(f"  ✅ Tool '{tool_name}' worked!")
                        print(f"     Response: {json.dumps(response.get('result', response))[:200]}...")
                        break
            
            if not found_tool:
                print(f"  ❌ No search tool available")
                print(f"     This indicates Rovo/MCP tools are not enabled")
            print()

print("=" * 70)

## 5. Comparison Test

Compare results from:
1. **Direct DC REST API** - Using existing implementation
2. **MCP via Cloud** - Via tunnel (if available)

This validates that the tunnel provides accurate results.

In [ ]:
# Import existing DC client for comparison
try:
    from src.confluence.rest_client import ConfluenceRestClient
    DC_CLIENT_AVAILABLE = True
    print("✅ DC REST client available for comparison")
except ImportError:
    DC_CLIENT_AVAILABLE = False
    print("❌ DC REST client not available")
    print("   Run from project root to import src modules")

In [ ]:
# Comparison test
print("=" * 70)
print("COMPARISON TEST: DC REST API vs MCP")
print("=" * 70)
print()

comparison_query = "documentation"  # Update with a query that should have results

# Test 1: Direct DC REST API
print("Test 1: Direct Data Center REST API")
if DC_CLIENT_AVAILABLE and config.has_dc_config():
    try:
        dc_client = ConfluenceRestClient(
            base_url=config.dc_url,
            username=config.dc_username,
            api_token=config.dc_api_token,
            verify_ssl=False,
        )
        
        import time
        start = time.time()
        dc_results = dc_client.search_pages(comparison_query, limit=5)
        dc_time = time.time() - start
        
        print(f"  ✅ Found {len(dc_results)} results in {dc_time:.2f}s")
        for page in dc_results[:3]:
            print(f"     - {page.get('title', 'No title')}")
    except Exception as e:
        print(f"  ❌ Error: {e}")
else:
    print("  ⏭️ Skipped - DC client or credentials not available")

print()

# Test 2: MCP via Cloud
print("Test 2: MCP via Cloud (Tunnel + Rovo)")
if config.has_cloud_config():
    headers = build_mcp_headers(config)
    
    init_result = send_mcp_request(
        config.mcp_endpoint,
        "initialize",
        {
            "protocolVersion": config.protocol_version,
            "capabilities": {"tools": {}},
            "clientInfo": {"name": "comparison-test", "version": "1.0.0"}
        },
        headers
    )
    
    if init_result["success"]:
        session_id = init_result.get("session_id")
        
        import time
        start = time.time()
        
        mcp_result = send_mcp_request(
            config.mcp_endpoint,
            "tools/call",
            {"name": "confluence_search", "arguments": {"query": comparison_query}},
            headers,
            session_id=session_id,
        )
        
        mcp_time = time.time() - start
        
        if mcp_result["success"] and "error" not in mcp_result["response"]:
            print(f"  ✅ MCP search completed in {mcp_time:.2f}s")
            print(f"     Response: {json.dumps(mcp_result['response'])[:200]}...")
        else:
            error = mcp_result.get("response", {}).get("error", {}).get("message", mcp_result.get("error"))
            print(f"  ❌ MCP search failed: {error}")
    else:
        print(f"  ❌ MCP init failed: {init_result.get('error')}")
else:
    print("  ⏭️ Skipped - Cloud credentials not configured")

print()
print("=" * 70)

## 6. Integration Patterns

Once the tunnel is working, here's how to integrate MCP into your RAG pipeline.

In [ ]:
# Example: MCP Client wrapper for RAG integration

class AtlassianMCPClient:
    """Client for accessing Confluence via Atlassian MCP server.
    
    Requires:
    - Atlassian Cloud subscription
    - Application Tunnel to Data Center (for DC content)
    - Rovo enabled and configured
    """
    
    def __init__(self, config: TunnelMCPConfig):
        self.config = config
        self.session_id = None
        self._headers = build_mcp_headers(config)
    
    def initialize(self) -> bool:
        """Initialize MCP session."""
        result = send_mcp_request(
            self.config.mcp_endpoint,
            "initialize",
            {
                "protocolVersion": self.config.protocol_version,
                "capabilities": {"tools": {}},
                "clientInfo": {"name": "rag-pipeline", "version": "1.0.0"}
            },
            self._headers
        )
        
        if result["success"]:
            self.session_id = result.get("session_id")
            return True
        return False
    
    def search(self, query: str, limit: int = 10) -> List[Dict[str, Any]]:
        """Search Confluence via MCP.
        
        Returns content from both Cloud and Data Center (if tunneled).
        """
        if not self.session_id:
            self.initialize()
        
        result = send_mcp_request(
            self.config.mcp_endpoint,
            "tools/call",
            {
                "name": "confluence_search",
                "arguments": {"query": query, "limit": limit}
            },
            self._headers,
            session_id=self.session_id,
        )
        
        if result["success"] and "error" not in result["response"]:
            return result["response"].get("result", {}).get("content", [])
        
        return []
    
    def get_page(self, page_id: str) -> Optional[Dict[str, Any]]:
        """Get a specific page by ID."""
        if not self.session_id:
            self.initialize()
        
        result = send_mcp_request(
            self.config.mcp_endpoint,
            "tools/call",
            {
                "name": "confluence_get_page",
                "arguments": {"page_id": page_id}
            },
            self._headers,
            session_id=self.session_id,
        )
        
        if result["success"] and "error" not in result["response"]:
            return result["response"].get("result")
        
        return None

print("AtlassianMCPClient class defined.")
print()
print("Example usage:")
print('  client = AtlassianMCPClient(config)')
print('  client.initialize()')
print('  results = client.search("project documentation")')

## 7. Results Summary

In [ ]:
print("=" * 70)
print("APPLICATION TUNNEL + MCP TEST SUMMARY")
print("=" * 70)
print(f"Timestamp: {datetime.now().isoformat()}")
print()

# Configuration status
print("Configuration:")
print(f"  Cloud Configured: {'✅' if config.has_cloud_config() else '❌'}")
print(f"  DC Configured: {'✅' if config.has_dc_config() else '❌'}")
print(f"  MCP SDK Available: {'✅' if MCP_AVAILABLE else '❌'}")
print()

# Prerequisites status
prereq_complete = sum(1 for p in prerequisites.values() if p["status"])
prereq_total = len(prerequisites)
print(f"Prerequisites: {prereq_complete}/{prereq_total} complete")
print()

# Next steps
print("Next Steps:")
if not config.has_cloud_config():
    print("  1. Acquire Atlassian Cloud subscription")
    print("  2. Configure Application Tunnel")
    print("  3. Enable Rovo and add DC as content source")
    print("  4. Add Cloud credentials to .env")
    print("  5. Re-run this notebook")
else:
    print("  1. Run Section 3 to test MCP connection")
    print("  2. If tools/list fails, verify Rovo + Tunnel setup with admin")
    print("  3. Run Section 4 to discover DC content")
    print("  4. Run Section 5 to compare with direct DC API")

print()
print("=" * 70)

## Appendix: Admin Setup Guide

Share this with your IT/Cloud administrators.

### Step 1: Atlassian Cloud Subscription
- Acquire Confluence Cloud (Premium or Enterprise tier for Rovo)
- Site URL will be: `https://yourcompany.atlassian.net`

### Step 2: Application Tunnel Setup
1. **On Data Center**: Install "Application Tunnels" from Atlassian Marketplace
2. **On Cloud**: Go to `admin.atlassian.com` → Security → Application Tunnels
3. Create new tunnel, generate security key
4. **On Data Center**: Add security key to tunnel client configuration
5. Verify tunnel shows "Connected" status

### Step 3: Rovo Configuration
1. Enable Rovo in Atlassian Cloud settings
2. Add Data Center as a connected content source
3. Select spaces to index
4. Wait for initial indexing to complete

### Step 4: MCP Server Access
1. Enable MCP server in Rovo settings
2. Configure allowed AI domains (if restricting access)
3. Generate API token for user access

### Documentation Links
- [Application Tunnels](https://support.atlassian.com/organization-administration/docs/connect-to-self-managed-products-with-application-tunnels/)
- [Rovo MCP Server](https://support.atlassian.com/rovo/docs/getting-started-with-the-atlassian-remote-mcp-server/)
- [Atlassian Cloud Admin](https://admin.atlassian.com)